In [ ]:
!pip install kagglehub
!pip install gensim
# =============================
# Imports
# =============================
import kagglehub
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# =============================
# Setup
# =============================
nltk.download('wordnet')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()

stop_words = set(stopwords.words('english'))
stop_words.discard('not')
stop_words.discard('no')
stop_words.discard('nor')

# =============================
# Load Dataset (FULL)
# =============================
path = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")
df = pd.read_csv(path + "/twitter_training.csv", header=None)
df.columns = ['id', 'entity', 'sentiment', 'text']

data = df[['text', 'sentiment']].dropna().copy()

# Optional: remove noisy label
data = data[data['sentiment'] != 'irrelevant']

# =============================
# Cleaning
# =============================
def clean_text(text):
    text = re.sub(r'<.*?>', '', str(text))
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(w, pos='v')
        for w in words
        if w not in stop_words
    ]

    return words

tokenized = [clean_text(t) for t in data['text']]
texts_joined = [" ".join(t) for t in tokenized]

# =============================
# TF-IDF (controlled size)
# =============================
tfidf = TfidfVectorizer(
    max_features=6000,   # keep under control
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(texts_joined)

# =============================
# Word2Vec
# =============================
w2v = Word2Vec(
    sentences=tokenized,
    vector_size=120,
    window=5,
    min_count=2,
    workers=2
)

def get_vector(words):
    vectors = [w2v.wv[w] for w in words if w in w2v.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(120)

X_w2v = np.array([get_vector(t) for t in tokenized])

# =============================
# Combine features
# =============================
X = np.hstack([X_w2v, X_tfidf.toarray()])
y = data['sentiment'].astype('category').cat.codes

# =============================
# Split (IMPORTANT)
# =============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =============================
# Random Forest
# =============================
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

# =============================
# ANN
# =============================
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

ann = MLPClassifier(
    hidden_layer_sizes=(256,128,64),
    max_iter=400,
    early_stopping=True,
    learning_rate='adaptive',
    random_state=42
)

ann.fit(X_train_s, y_train)
ann_pred = ann.predict(X_test_s)

print("ANN Accuracy:", accuracy_score(y_test, ann_pred))

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   --- ------------------------------------ 2.4/24.4 MB 13.4 MB/s eta 0:00:02
   -------- ------------------------------- 5.2/24.4 MB 13.7 MB/s eta 0:00:02
   ------------- -------------------------- 8.4/24.4 MB 13.9 MB/s eta 0:00:02
   ------------------ --------------------- 11.0/24.4 MB 13.4 MB/s eta 0:00:02
   --------------------------- ------------ 16.5/24.4 MB 16.0 MB/s eta 0:00:01
   --------------------------------- ------ 20.2/24.4 MB 16.4 MB/s eta 0:00:01
   ---------------------------------------  24.1/24.4 MB 16.6 MB/s eta 0:00:01
   ---------------------------------------  24.4/24.4 MB 16.5 MB/s eta 0:00:01
   ---------------------------------------  24.4/24.4 MB 16.5 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 12.2 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [smart_open]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- 

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\vatsu\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vatsu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


100%|█████████████████████████████████████████████████████████████████████████████| 1.99M/1.99M [00:01<00:00, 1.31MB/s]

Extracting files...



Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Random Forest Accuracy: 0.7442567567567567


## Random Forest Accuracy: 0.7442567567567567
## ANN Accuracy: 0.8472972972972973